In [51]:
import requests
import pandas
import os
import time
import json
from bs4 import BeautifulSoup

In [ ]:
headers = {'User-Agent': 'XXX@mail.com'}

In [53]:
cik = "0001708176"
url = f"https://data.sec.gov/submissions/CIK{cik}.json"
company_filings = requests.get(url, headers=headers)


In [54]:
SAVE_DIR = "sec_filings_html_test"
os.makedirs(SAVE_DIR, exist_ok=True)

# ===== 获取申报清单 =====
resp = requests.get(url, headers=headers)
if resp.status_code != 200:
    print(f"获取申报 JSON 失败: HTTP {resp.status_code}")
    raise SystemExit()

data = resp.json()
filings = data.get("filings", {}).get("recent", {})
accessions = filings.get("accessionNumber", [])
forms = filings.get("form", [])
docs = filings.get("primaryDocument", [])
dates = filings.get("filingDate", [])


# ===== 下载主文件 HTML =====
def download_html(acc, form, doc, fdate):
    # 构造 URL
    acc_no_dashless = acc.replace("-", "")
    base_url = f"https://www.sec.gov/Archives/edgar/data/{int(cik)}/{acc_no_dashless}/"
    file_url = base_url + doc

    # 清理文件名，例如 “8-K_2023-06-15_0001708176-23-000045.htm”
    safe_form = form.replace("/", "_")
    safe_filename = f"{safe_form}_{fdate}_{acc}.htm"
    out_path = os.path.join(SAVE_DIR, safe_filename)

    # 如果已经存在，就跳过
    if os.path.exists(out_path):
        print("✔ 已存在:", out_path)
        return

    # 发请求下载
    r = requests.get(file_url, headers=headers)
    if r.status_code == 200:
        with open(out_path, "wb") as f:
            f.write(r.content)
        print("✅ 下载成功:", out_path)
    else:
        print("❌ 下载失败:", file_url, "状态码:", r.status_code)
        # 如果 status_code 是 403 或其它，可能是被 SEC 拦截


# ===== 主流程 =====
for acc, form, doc, fdate in zip(accessions, forms, docs, dates):
    # 下载所有文件
    try:
        download_html(acc, form, doc, fdate)
    except Exception as e:
        print("异常:", acc, form, e)

    time.sleep(0.5)  # limit rate

print("下载测试完成, 存在目录:", SAVE_DIR)

✅ 下载成功: sec_filings_html_test/SCHEDULE 13D_A_2025-09-09_0001213900-25-086245.htm
✅ 下载成功: sec_filings_html_test/8-K_2025-09-09_0001140361-25-034348.htm
✅ 下载成功: sec_filings_html_test/8-K_2025-09-09_0001140361-25-034348.htm
✅ 下载成功: sec_filings_html_test/10-Q_2025-08-12_0001213900-25-074948.htm
✅ 下载成功: sec_filings_html_test/10-Q_2025-08-12_0001213900-25-074948.htm
✅ 下载成功: sec_filings_html_test/SC 13E3_A_2025-08-11_0001213900-25-073796.htm
✅ 下载成功: sec_filings_html_test/SC 13E3_A_2025-08-11_0001213900-25-073796.htm
✅ 下载成功: sec_filings_html_test/DEFM14A_2025-08-08_0001140361-25-029900.htm
✅ 下载成功: sec_filings_html_test/DEFM14A_2025-08-08_0001140361-25-029900.htm
✅ 下载成功: sec_filings_html_test/8-K_2025-08-01_0001140361-25-028412.htm
✅ 下载成功: sec_filings_html_test/8-K_2025-08-01_0001140361-25-028412.htm
✅ 下载成功: sec_filings_html_test/SC 13E3_A_2025-07-25_0001213900-25-067818.htm
✅ 下载成功: sec_filings_html_test/SC 13E3_A_2025-07-25_0001213900-25-067818.htm
✅ 下载成功: sec_filings_html_test/PRER14A_2025-07